In [2]:
#importing libraries
import os 
import pandas as pd 
import numpy as np 

import torch 
import torchvision.models as models
import torchvision.transforms as transforms 

from PIL import Image 
 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score

In [3]:
#setting path 
data_dir = "CBD_Coffee Bean Dataset/CBD_Coffee Bean Dataset"

In [4]:
#loading pretrained resnet18
model = models.resnet18(pretrained=True)

#removing last layer to get features only
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()

/opt/anaconda3/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [5]:
#image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [6]:
#function to extract features 
def get_features(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad(): 
        features = model(img)

    return features.flatten().numpy()

In [7]:
#loading images and labels from folders
X = []
y = []


#looping through each folder 
for label in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, label)

    if os.path.isdir(folder_path):

        for file in os.listdir(folder_path):

            img_path = os.path.join(folder_path, file)

            try:
                feats = get_features(img_path)
                X.append(feats)
                y.append(label)
            except:
                pass

In [8]:
#train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
#training the model 
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [10]:
#evaluating
preds = model_lr.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("Image Model Accuracy", accuracy)

Image Model Accuracy 0.8064516129032258


In [ ]:
#controlled experiment / example of ablation
import os
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data_dir = "CBD_Coffee Bean Dataset/CBD_Coffee Bean Dataset"

def get_features(img):
    img = img.resize((224, 224))
    return np.array(img).flatten()

#testing different dataset sizes
for num in [10, 20, 30]:

    X = []
    y = []

    for label in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, label)

        if os.path.isdir(folder_path):
            for i, file in enumerate(os.listdir(folder_path)):
                if i >= num:
                    break

                path = os.path.join(folder_path, file)

                try:
                    img = Image.open(path).convert("RGB")
                    feats = get_features(img)

                    X.append(feats)
                    y.append(label)

                except:
                    pass

    X = np.array(X)
    y = np.array(y)

    if len(X) == 0:
        break

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print(f"Images per class: {num}, Accuracy: {acc:.3f}")


Images per class: 10, Accuracy: 0.556
Images per class: 20, Accuracy: 0.278
Images per class: 30, Accuracy: 0.389
